### Load Covariance Matrix

In [6]:
import numpy as np
import pytest
import pyccl as ccl
from pyccl import CCLWarning

from pyccl.pk2d import Pk2D
from pyccl.correlations import correlation

#load covariance matrix and dv from cosmolike
cov_file = './cocoa/Cocoa/projects/lsst_fourier/data/cov_lsst_fourier'
mask_file = './cocoa/Cocoa/projects/lsst_fourier/data/lsst_Y3.mask'
mask = np.loadtxt(mask_file)[:,1].astype(bool)
cov_raw = np.loadtxt(cov_file)
ncov = int(np.max(cov_raw[:,0]))+1
cov = np.zeros((ncov, ncov))
for i in range(len(cov_raw)):
    ii = int(cov_raw[i, 0])
    jj = int(cov_raw[i, 1])
    element = cov_raw[i,8] + cov_raw[i,9]
    cov[ii,jj] = element
    cov[jj,ii] = element

### COMPARE SETUP0

In [7]:
#------------------INPUT------------------------------------------------------#
SETUP = 'setup0'
#-----------------------------------------------------------------------------#
#---------YOU STILL NEED TO MODIFY THE CCL'S MODELLING BELOW------------------#
#-----------------------------------------------------------------------------#
# ccl.gsl_params.LENSING_KERNEL_SPLINE_INTEGRATION = False
# setup cosmology
COSMO = ccl.Cosmology(
    Omega_c=0.26507647072945384,
    Omega_b=0.0495,
    Omega_k=0,
    h=0.6732,
    w0=-1,
    wa=0,
    A_s=2.1/1e9,
    n_s=0.96605,
    m_nu=0.06,
    Neff=3.046,
    mass_split='single',
    transfer_function='boltzmann_camb',
    matter_power_spectrum='camb',
    extra_parameters = {"camb": {"halofit_version": "takahashi",
                                 'AccuracyBoost': 1.0,
                                 'kmax':15,
                                 'dark_energy_model': 'ppf',
                                 'accurate_massive_neutrino_transfer': False,
                                 'k_per_logint': 15,
                                 }}
    )
h=0.6732

#-----------------------------------------------------------------#
#---------------based on discussion in section 0------------------#
#------------------we use this power spectrum---------------------#
#-----------------------------------------------------------------# 

filepath = f'./cocoa/Cocoa/projects/lsst_fourier/chains/lsst_{SETUP}_evaluate/'
z_pk_cocoa = np.loadtxt(filepath+'z_pk_1.txt')
k_pk_cocoa = np.loadtxt(filepath+'k_pk_1.txt')   #1/Mpc
lnPk_cocoa = np.loadtxt(filepath+'pknl_1.txt').reshape(len(k_pk_cocoa),len(z_pk_cocoa) ).T   #Mpc/h^3
lnPk_lin_cocoa = np.loadtxt(filepath+'pkln_1.txt').reshape(len(k_pk_cocoa),len(z_pk_cocoa) ).T   #Mpc/h^3
lnPk_cocoa = lnPk_cocoa[::-1,:] - 3*np.log(h)
lnPk_lin_cocoa = lnPk_lin_cocoa[::-1,:] - 3*np.log(h)
a_pk_cocoa = 1/(1+z_pk_cocoa)[::-1]
lnk_pk_cocoa = np.log(k_pk_cocoa)

pk2_cocoa = Pk2D(a_arr=a_pk_cocoa, lk_arr=lnk_pk_cocoa, pk_arr=lnPk_cocoa, is_logp=True)
pk2_lin_cocoa = Pk2D(a_arr=a_pk_cocoa, lk_arr=lnk_pk_cocoa, pk_arr=lnPk_lin_cocoa, is_logp=True)
pk2_ccl     = COSMO.nonlin_matter_power(k = np.exp(lnk_pk_cocoa), a = a_pk_cocoa)
pk2_lin_ccl = COSMO.linear_matter_power(k = np.exp(lnk_pk_cocoa), a = a_pk_cocoa)
pk2_ccl     = Pk2D(a_arr=a_pk_cocoa, lk_arr=lnk_pk_cocoa, pk_arr=np.log(pk2_ccl), is_logp=True)
pk2_lin_ccl = Pk2D(a_arr=a_pk_cocoa, lk_arr=lnk_pk_cocoa, pk_arr=np.log(pk2_lin_ccl), is_logp=True)

#---------------------------------------------------------------------------------------------#
#-------------------this Pk is further injected into power specetrum calculation--------------#
#---------------------------------------------------------------------------------------------#


#set up lens source sample
srcs_nzs = np.loadtxt('./cocoa/Cocoa/projects/lsst_fourier/data/lsst_srcs_4cosmolike.nz')
lens_nzs = np.loadtxt('./cocoa/Cocoa/projects/lsst_fourier/data/lsst_lens_4cosmolike.nz')

z_srcs = srcs_nzs[:,0]+0.005
srcs_nz = srcs_nzs[:,1:]
z_lens = lens_nzs[:,0]+0.005
lens_nz = lens_nzs[:,1:]

srcs = []
lens = []
nsrcs = 5
nlens = 10
gbias =  [1.09,1.15,1.21,1.27,1.33,1.40,1.46,1.53,1.60,1.67]
ggl_exclude = []

for i in range(nsrcs):
    srcs.append(ccl.WeakLensingTracer(COSMO, dndz=(z_srcs, srcs_nz[:,i]), has_shear=True, n_samples=400))
for i in range(nlens):
    lens.append(ccl.NumberCountsTracer(COSMO, dndz=(z_lens, lens_nz[:,i]), bias=(z_lens,np.ones_like(z_lens)*gbias[i]), has_rsd=False, n_samples=400))


#calculate the power spectrum
ncl= 20
lmin = 20
lmax = 4000
logdl = (np.log(lmax) - np.log(lmin))/ncl
ells = np.zeros(int(ncl))
for i in range(int(ncl)):
    ells[i] = np.exp(np.log(lmin) + (i + 0.5)*logdl)

corrs = []
for i in range(nsrcs):
    for j in range(i,nsrcs):
        corrs.append(ccl.angular_cl(COSMO, srcs[i], srcs[j], ells, p_of_k_a=pk2_cocoa, l_limber=-1) )
        
for i in range(nlens):
    for j in range(nsrcs):
        if [i,j] in ggl_exclude:
            continue
        corrs.append(ccl.angular_cl(COSMO, lens[i], srcs[j], ells, p_of_k_a=pk2_cocoa, l_limber=-1) )
        
for i in range(nlens):
    corrs.append(ccl.angular_cl(COSMO, lens[i], lens[i], ells, p_of_k_a=pk2_cocoa, l_limber=-1))
    
dv_ccl = np.concatenate((corrs))

dv_cosmolike = np.loadtxt(f'./cocoa/Cocoa/projects/lsst_fourier/chains/lsst_{SETUP}_evaluate/lsst.modelvector_1')[:,1]

import importlib
import utils
importlib.reload(utils)

utils.compare(
    dv_ccl,
    dv_cosmolike,
    cov,
    mask,
    'ccl',
    'cosmolike',
    'fourier',
    10,
    5,
    20,
    4000,
    20,
    [True, True, True],
    [True, True, True],
    [True, True, True],
    True,
    True,
    True,
    False,
)

ss chi2 is 0.003/0.016
gs chi2 is 0.007/0.033
gg chi2 is 0.008/0.161
3x2pt chi2 is 0.020/0.224
